In [14]:
# ===========================================
# 1. importaciones
# ===========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from optbinning import BinningProcess, Scorecard
from optbinning.scorecard import plot_auc_roc, plot_cap, plot_ks

# Definición de parámetros en un diccionario
binning_fit_params = {
    "default": { # Aplica a todas las variables por defecto
        "min_bin_n_nonevent": 5,
        "min_bin_n_event": 5  
    }
}

# Configuración
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print(f"✅ Librerias cargadas correctamente")

✅ Librerias cargadas correctamente


In [8]:
# ===========================================
# 2. Carga de datos
# ===========================================
df = pd.read_csv(r"D:\05 - Modelo Scoring Crediticion\Codigo\scoring_crediticio\data\raw\scoring_crediticio.csv")

# Elimar el ID (no es predictivo)
df = df.drop(columns=['id_solicitud'])

X = df.drop(columns=['default'])
y = df['default']

# ⚠️ CRITICO: Split ANTES de WoE (Weight of Evidence)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y) # lo mantenemos proporcional al default

print("=" * 60)
print("DIVISION TRAIN / TEST")
print(f"Train: {len(X_train):,} | default: {y_train.mean():.2%}")
print(f"Test: {len(X_test):,} | default: {y_test.mean():.2%}")
print("=" * 60)

DIVISION TRAIN / TEST
Train: 700 | default: 30.14%
Test: 300 | default: 30.00%


In [25]:
# ===========================================
# 3. Configurar BinningProcess
# ===========================================

# Lista de variables a procesar
variable_name = X_train.columns.tolist()

print("Variables a procesar:")
for v in variable_name:
    print(f" - {v}")
print("=" * 60)

# Criterios de selección: solo varibles con IV > 0.02
selection_criteria = {"iv": {"min": 0.02, "max": 1.0}}

# Crear el proceso de binning
binning_process = BinningProcess(
    variable_names = variable_name,
    selection_criteria = selection_criteria,
    # Parámetros opcionales recomendadas para scoring
    max_n_bins=8,           # Máximo 8 bins por variable
    min_bin_size=0.05,        # Tamaño mínimo de cada bin
    binning_fit_params=binning_fit_params # Pasan los parametros binning_fit_params
)

binning_process.fit(X_train, y_train)

print("\n BinningProcess configurado correctamente.")

Variables a procesar:
 - edad
 - ciudad
 - estrato
 - ingreso_mensual
 - antiguedad_laboral_meses
 - num_creditos_activos
 - utilizacion_cupo
 - mora_maxima_12m
 - num_consultas_6m
 - dti

 BinningProcess configurado correctamente.


In [26]:
# =============================================
# 4. Entrenar BinningProcess
# =============================================

print("Binning entrenado")
print(f"\n Variables seleccionadas (IV > 0.02):")
for v in binning_process.variable_names:
    print(f" - {v}")

# Ver variables descartadas (IV <= 0.02)
all_vars =  set(variable_name)
mark = binning_process.variable_names
selected_vars = set([v for v, keep in zip(variable_name, mark) if keep])
discarded = all_vars - selected_vars

if discarded:
    print(f"\n Variables descartadas (IV <= 0.02):")
    for v in discarded:
        print(f" - {v}")

Binning entrenado

 Variables seleccionadas (IV > 0.02):
 - edad
 - ciudad
 - estrato
 - ingreso_mensual
 - antiguedad_laboral_meses
 - num_creditos_activos
 - utilizacion_cupo
 - mora_maxima_12m
 - num_consultas_6m
 - dti


In [30]:
# =============================================
# 5. Ver detalle del binning (WoE, IV)
# =============================================

# Analizar 'utilizacion_cupo'
optb = binning_process.get_binned_variable('utilizacion_cupo')
tabla = optb.binning_table.build()

print("Binning de 'utilizacion_cupo':")
display(tabla)

# Extraer el IV
print(f"IV de 'utilizacion_cupo': {optb.binning_table.iv:.4f}")

Binning de 'utilizacion_cupo':


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.20)",233,0.332857,161,72,0.309013,-0.035766,0.000429,0.000054
1,"[0.20, 0.23)",54,0.077143,39,15,0.277778,0.115007,0.000996,0.000124
2,"[0.23, 0.25)",36,0.051429,29,7,0.194444,0.580881,0.015178,0.001871
3,"[0.25, 0.27)",37,0.052857,31,6,0.162162,0.801723,0.028027,0.003412
4,"[0.27, 0.34)",87,0.124286,64,23,0.264368,0.182885,0.004001,0.000499
5,"[0.34, inf)",253,0.361429,165,88,0.347826,-0.211896,0.016875,0.002105
6,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
7,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
Totals,,700,1.000000,489,211,0.301429,,0.065506,0.008066


IV de 'utilizacion_cupo': 0.0655


In [ ]:
# =============================================
# 6. Gráfico de WoE de una variable
# =============================================

# Asegurar que la figura se muestre en VS Code / Jupyter
%matplotlib inline

# Crear la tabla de bins para graficar sin depender del plot interno de optbinning
woe_table = optb.binning_table.build().copy()
woe_table = woe_table[~woe_table['Bin'].isin(['Special', 'Missing', 'Totals'])].copy()
woe_table['WoE'] = pd.to_numeric(woe_table['WoE'], errors='coerce')
woe_table['Event rate'] = pd.to_numeric(woe_table['Event rate'], errors='coerce')
woe_table = woe_table.dropna(subset=['WoE', 'Event rate']).reset_index(drop=True)

# 1) WoE por bin
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(range(len(woe_table)), woe_table['WoE'], marker='o', linestyle='-', linewidth=2, color='#2E86DE')
ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.7)
ax.set_xticks(range(len(woe_table)))
ax.set_xticklabels(woe_table['Bin'], rotation=45, ha='right')
ax.set_title('WoE de utilizacion_cupo', fontweight='bold')
ax.set_xlabel('Bins')
ax.set_ylabel('WoE')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Guardar imagen
fig.savefig(r'D:\05 - Modelo Scoring Crediticion\Codigo\scoring_crediticio\reports\06_woe_utilizacion.png', dpi=100, bbox_inches='tight')

# 2) Tasa de default por bin
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(woe_table)), woe_table['Event rate'], color='#FF7F50', edgecolor='black')
ax.set_xticks(range(len(woe_table)))
ax.set_xticklabels(woe_table['Bin'], rotation=45, ha='right')
ax.set_title('Tasa de default por Bin - utilizacion_cupo', fontweight='bold')
ax.set_xlabel('Bins')
ax.set_ylabel('Event rate')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

fig.savefig(r'D:\05 - Modelo Scoring Crediticion\Codigo\scoring_crediticio\reports\07_eventrate_utilizacion.png', dpi=100, bbox_inches='tight')